In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import mne
import eelbrain
from eelbrain import *

## User settings

In [ ]:
MAINDIR = Path(r"C:/projects/emo_EEG")
EEG_DIR = MAINDIR / "data_pipeline" / "mTRF" / "eelbrain"/ "data"
PREDICTOR_DIR = MAINDIR / "data_pipeline" / "mTRF" / "eelbrain"/ "predictors"
OUT_DIR = MAINDIR / "data_pipeline" / "mTRF" / "eelbrain"/ "data"/ "trf"
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_ORDER = 1
EPOCH_DONE = True
T_START = -0.1 # in second
T_END = 0.5 
BASIS = 0.05
PARTITION = 4
SCALE_METHOD = 'l1' # l1: divide by std; l2: divide by mean absolute value
SUBS = [f"sub-pilot_{i}" for i in range(1, 8)]
# SUBS = ["sub-pilot_Jun"]
PREDICTORS = ['MFCCs']

## Helpers

In [3]:
def add_condition_metadata(ds, stim_col='stim_name', eeg_col='eeg'):
    stim_names = list(ds[stim_col])

    gender = []
    speech_style = []
    emotion = []
    duration = []

    for name, nd in zip(stim_names, ds[eeg_col]):
        parts = name.replace('.wav', '').split('_')

        # expected: Gender_Style_Emotion_cont_...
        gender.append(parts[0])
        speech_style.append(parts[1])
        emotion.append(parts[2])

        # duration from EEG NDVar length
        duration.append(len(nd) * nd.time.tstep)

    ds = ds.copy()
    ds['gender'] = Factor(gender)
    ds['speech_style'] = Factor(speech_style)
    ds['emotion'] = Factor(emotion)
    ds['duration'] = Var(duration)

    return ds

def concat_ndvars_time(ndvars):
    if len(ndvars) == 0:
        raise ValueError("ndvars is empty")

    first = ndvars[0]
    dim_names_first = [dim.name for dim in first.dims]
    time_axis = dim_names_first.index('time')

    #print("FIRST:")
    #print("  dims:", dim_names_first)
    #print("  shape:", first.x.shape)

    for i, nd in enumerate(ndvars[1:], start=1):
        dim_names = [dim.name for dim in nd.dims]
        #print(f"ITEM {i}:")
        #print("  dims:", dim_names)
        #print("  shape:", nd.x.shape)

        if dim_names != dim_names_first:
            raise ValueError(
                f"Dimension name/order mismatch at item {i}: "
                f"{dim_names} vs {dim_names_first}"
            )

        for ax, (dim_a, dim_b) in enumerate(zip(nd.dims, first.dims)):
            if ax == time_axis:
                continue
            if len(dim_a) != len(dim_b):
                raise ValueError(
                    f"Non-time dim length mismatch at item {i}, "
                    f"dim '{dim_a.name}': {len(dim_a)} vs {len(dim_b)}"
                )

    x = np.concatenate([nd.x for nd in ndvars], axis=time_axis)
    total_n = sum(len(nd.time) for nd in ndvars)
    new_time = UTS(first.time.tmin, first.time.tstep, total_n)

    new_dims = list(first.dims)
    new_dims[time_axis] = new_time

    return NDVar(x, new_dims, name=first.name, info=first.info)

def concat_by_emotion(ds, signal_key, stim_name_key='stim_name'):
    emotions_unique = list(dict.fromkeys(ds['emotion']))
    ds_list = []

    for emo in emotions_unique:
        sub = ds[ds['emotion'] == emo]
        if len(sub) == 0:
            continue

        nds = list(sub[signal_key])
        signal_cat = concat_ndvars_time(nds)

        row = sub[:1].copy()
        row[signal_key] = [signal_cat]

        # update metadata
        row['stim_idx'] = [-1]

        if stim_name_key in row:
            row[stim_name_key] = [f"{emo}_concat"]

        if 'gender' in row:
            row['gender'] = ['mixed']

        if 'speech_style' in row:
            row['speech_style'] = ['mixed']

        if 'duration' in row:
            row['duration'] = [sum(sub['duration'])]

        ds_list.append(row)

    return eelbrain.combine(ds_list)

def get_row_by_emotion(ds, emotion):
    sub = ds[ds['emotion'] == emotion]
    if sub.n_cases != 1:
        raise ValueError(f"Expected exactly 1 row for emotion={emotion}, got {sub.n_cases}")
    return sub

def get_predictor_ndvars(ds_pre_concat_sub, emotion, predictor_keys):
    x_list = []

    for key in predictor_keys:
        row = get_row_by_emotion(ds_pre_concat_sub[key], emotion)
        x_list.append(row[key][0])

    return x_list[0] if len(x_list) == 1 else x_list
    
def crop_ndvar_to_n_samples(nd, n):
    cropped = nd.sub(time=nd.time[:n])
    if len(cropped.time) != n:
        raise ValueError(
            f"Cropping failed: requested {n}, got {len(cropped.time)}"
        )
    return cropped

def align_trial_sets(ds_eeg, ds_predictors, predictor_keys):
    ds_eeg = ds_eeg.copy()
    ds_predictors = {k: ds.copy() for k, ds in ds_predictors.items()}

    n_cases = ds_eeg.n_cases
    for key in predictor_keys:
        if ds_predictors[key].n_cases != n_cases:
            raise ValueError(
                f"Case mismatch for {key}: EEG={n_cases}, PRE={ds_predictors[key].n_cases}"
            )

    eeg_aligned = []
    pred_aligned = {k: [] for k in predictor_keys}

    for i in range(n_cases):
        eeg_name = ds_eeg['stim_name'][i]
        eeg_nd = ds_eeg['eeg'][i]

        # make sure EEG has time dimension
        if not hasattr(eeg_nd, 'time'):
            raise ValueError(f"EEG row {i} has no time dimension")

        lengths = [len(eeg_nd.time)]

        for key in predictor_keys:
            pred_ds = ds_predictors[key]
            pred_nd = pred_ds[key][i]
            pre_name = pred_ds['filename'][i]

            if eeg_name != pre_name:
                raise ValueError(
                    f"Filename mismatch at row {i}, key={key}: EEG={eeg_name}, PRE={pre_name}"
                )

            # works for both 1D and 2D NDVars, as long as they have a time dimension
            if not hasattr(pred_nd, 'time'):
                raise ValueError(
                    f"Predictor '{key}' at row {i} has no time dimension"
                )

            lengths.append(len(pred_nd.time))

        n = min(lengths)

        # crop EEG along time only
        eeg_aligned.append(crop_ndvar_to_n_samples(eeg_nd, n))

        # crop each predictor along time only
        for key in predictor_keys:
            pred_nd = ds_predictors[key][key][i]
            pred_aligned[key].append(crop_ndvar_to_n_samples(pred_nd, n))

    ds_eeg['eeg'] = eeg_aligned
    for key in predictor_keys:
        ds_predictors[key][key] = pred_aligned[key]

    return ds_eeg, ds_predictors

def get_eeg_ndvar(ds_eeg_concat_sub, emotion):
    row = get_row_by_emotion(ds_eeg_concat_sub, emotion)
    return row['eeg'][0]

## data loading and preparating

In [ ]:
# load predictor datasets
ds_predictors = {}
for key in PREDICTORS:
    ds_pre = eelbrain.load.unpickle(PREDICTOR_DIR / f'emo_pilot_order{N_ORDER}_{key}.pickle')
    ds_predictors[key] = ds_pre

# load eeg data
ds_eegs = {}
for sub in SUBS:
    ds_eeg = eelbrain.load.unpickle(EEG_DIR / f'{sub}_eeg_acq-2.pickle')
    ds_eeg = add_condition_metadata(ds_eeg, stim_col='stim_name', eeg_col='eeg')
    ds_eegs[sub] = ds_eeg

ds_eegs_aligned = {}
ds_predictors_aligned = {}

for sub, ds_eeg in ds_eegs.items():
    ds_eeg_aligned, ds_pre_aligned = align_trial_sets(ds_eeg, ds_predictors, PREDICTORS)
    ds_eegs_aligned[sub] = ds_eeg_aligned
    ds_predictors_aligned[sub] = ds_pre_aligned

ds_eegs_concat = {}
ds_pre_concat = {}

for sub in SUBS:
    ds_eegs_concat[sub] = concat_by_emotion(
        ds_eegs_aligned[sub],
        signal_key='eeg',
        stim_name_key='stim_name'
    )

    ds_pre_concat[sub] = {}
    for key in PREDICTORS:
        ds_pre_concat[sub][key] = concat_by_emotion(
            ds_predictors_aligned[sub][key],
            signal_key=key,
            stim_name_key='filename'
        )

## Estimate TRFs

In [6]:
models = {
    'MFCCs': ['MFCCs']
}
emotions = ['hap', 'sad'] 

trf_results = {}
# Loop over subjects
for sub, ds_eeg_sub in ds_eegs_concat.items():
    trf_results[sub] = {}
    # Loop over selected emotions
    for emotion in emotions:
        trf_paths = {model: OUT_DIR / f'{sub}_{emotion}_{model}.pickle' for model in models}
        trf_results[sub][emotion] = {}
        y = get_eeg_ndvar(ds_eegs_concat[sub], emotion)
        # Loop over selected models
        for model_name, predictor_keys in models.items():
            path = trf_paths[model_name]
            # Skip if this file already exists
            if path.exists():
                continue
            print(f"Estimating: {sub} ~ {emotion} ~ {model_name}")
            x = get_predictor_ndvars(ds_pre_concat[sub], emotion, predictor_keys)
            result = eelbrain.boosting(
                y=y,
                x=x,
                tstart=T_START,
                tstop=T_END,
                basis=BASIS,         
                partitions=PARTITION,   
                test=1,               
                selective_stopping=True,
                scale_data=True,
                error = SCALE_METHOD
            )
            eelbrain.save.pickle(result, path)

Estimating: sub-pilot_Jun ~ hap ~ MFCCs
Estimating: sub-pilot_Jun ~ sad ~ MFCCs


In [ ]:
print(result.r)
print(result.r.x)
print(result.r.sensor.names)

<NDVar 'Correlation': 32 sensor>
[ 0.00726319  0.0556752   0.04336746  0.06392751  0.06707348  0.04555825
  0.02616834  0.0420755   0.03598976  0.02557643 -0.01132164  0.02374556
  0.02068381  0.01823782  0.01565104  0.0136078   0.01868029  0.02072594
  0.01696388  0.02141833  0.02389763  0.03309522  0.0439826   0.01986126
  0.06933515  0.07016883  0.07557715  0.07881482  0.07518005  0.05214831
  0.07541039  0.04729475]
['Fp1', 'AF3', 'F7', 'F3', 'FC1', 'FC5', 'T7', 'C3', 'CP1', 'CP5', 'P7', 'P3', 'Pz', 'PO3', 'O1', 'Oz', 'O2', 'PO4', 'P4', 'P8', 'CP6', 'CP2', 'C4', 'T8', 'FC6', 'FC2', 'F4', 'F8', 'AF4', 'Fp2', 'Fz', 'Cz']


In [ ]:
def compare_trial_lengths(ds_eeg, ds_pre, emotion, pre_key='envelope'):
    eeg_sub = ds_eeg[ds_eeg['emotion'] == emotion]
    pre_sub = ds_pre[pre_key][ds_pre[pre_key]['emotion'] == emotion]

    print(f"EEG cases: {eeg_sub.n_cases}")
    print(f"PRE cases: {pre_sub.n_cases}")

    if eeg_sub.n_cases != pre_sub.n_cases:
        print("Case count mismatch!")
        return

    for i in range(eeg_sub.n_cases):
        eeg_name = eeg_sub['stim_name'][i] if 'stim_name' in eeg_sub else eeg_sub['filename'][i]
        pre_name = pre_sub['filename'][i]
        eeg_len = len(eeg_sub['eeg'][i].time)
        pre_len = len(pre_sub[pre_key][i].time)
        diff = eeg_len - pre_len
        print(i, eeg_name, pre_name, eeg_len, pre_len, diff)

In [ ]:
compare_trial_lengths(ds_eegs_aligned['sub-pilot_1'], ds_pre_aligned, 'hap', 'gammatone_8')

EEG cases: 16
PRE cases: 16
0 Fem_ADS_hap_cont_1_scaled.wav Fem_ADS_hap_cont_1_scaled.wav 6584 6583 1
1 Fem_ADS_hap_cont_2_scaled.wav Fem_ADS_hap_cont_2_scaled.wav 6584 6583 1
2 Fem_ADS_hap_cont_3_scaled.wav Fem_ADS_hap_cont_3_scaled.wav 6584 6583 1
3 Fem_ADS_hap_cont_4_scaled.wav Fem_ADS_hap_cont_4_scaled.wav 6584 6583 1
4 Fem_CDS_hap_cont_1_scaled.wav Fem_CDS_hap_cont_1_scaled.wav 6554 6553 1
5 Fem_CDS_hap_cont_2_scaled.wav Fem_CDS_hap_cont_2_scaled.wav 6554 6553 1
6 Fem_CDS_hap_cont_3_scaled.wav Fem_CDS_hap_cont_3_scaled.wav 6554 6553 1
7 Fem_CDS_hap_cont_4_scaled.wav Fem_CDS_hap_cont_4_scaled.wav 6554 6553 1
8 Male_ADS_hap_cont_1_scaled.wav Male_ADS_hap_cont_1_scaled.wav 7071 7070 1
9 Male_ADS_hap_cont_2_scaled.wav Male_ADS_hap_cont_2_scaled.wav 7071 7070 1
10 Male_ADS_hap_cont_3_scaled.wav Male_ADS_hap_cont_3_scaled.wav 7071 7070 1
11 Male_ADS_hap_cont_4_scaled.wav Male_ADS_hap_cont_4_scaled.wav 7071 7070 1
12 Male_CDS_hap_cont_1_scaled.wav Male_CDS_hap_cont_1_scaled.wav 6185 6185

In [ ]:
print(ds_pre_aligned['gammatone_8'])
print(ds_eegs_aligned['sub-pilot_1'])

#    stim_idx   filename                         gender   speech_style   emotion   duration   gammatone_8    
-------------------------------------------------------------------------------------------------------------
0    1          Fem_ADS_hap_cont_1_scaled.wav    Fem      ADS            hap       51.43      <NDVar 'Fem_...
1    2          Fem_ADS_hap_cont_2_scaled.wav    Fem      ADS            hap       51.43      <NDVar 'Fem_...
2    3          Fem_ADS_hap_cont_3_scaled.wav    Fem      ADS            hap       51.43      <NDVar 'Fem_...
3    4          Fem_ADS_hap_cont_4_scaled.wav    Fem      ADS            hap       51.43      <NDVar 'Fem_...
4    5          Fem_ADS_sad_cont_1_scaled.wav    Fem      ADS            sad       58.625     <NDVar 'Fem_...
5    6          Fem_ADS_sad_cont_2_scaled.wav    Fem      ADS            sad       58.625     <NDVar 'Fem_...
6    7          Fem_ADS_sad_cont_3_scaled.wav    Fem      ADS            sad       58.625     <NDVar 'Fem_...
7    8    